# 17 — Pauli, Freeman–Durden & Yamaguchi

Three classic decompositions, all run on the same subset as everything else in this branch. Each one reports a reconstruction residual, so you can see where the model assumptions break down instead of trusting every pixel blindly.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from nisar_utils.config import load_config
from nisar_utils.polarimetric_io import load_detected_mode, load_branch_terms, FULL_POL_TERMS
from nisar_utils.polarimetry import *
cfg=load_config(); mode=load_detected_mode(cfg)
if mode['polarimetric_mode']!='LSAR_FULL_POL': print('SKIPPED:',mode['polarimetric_mode'])
else:
    data,sp,guard,mode=load_branch_terms(cfg,FULL_POL_TERMS)
    C=covariance_hermitian_from_terms(data)
    print('Exact persisted subset:',sp,'shape=',C.shape[:-2])
    pauli=pauli_powers(C)
    print('Pauli span median:',float(np.nanmedian(pauli['span'])))
    fd=fullpol_freeman_durden(C)
    y4=fullpol_yamaguchi4(C)
    print('Freeman–Durden residual RMS:',float(np.sqrt(np.nanmean(fd['residual']**2))))
    print('Yamaguchi-4 residual RMS:',float(np.sqrt(np.nanmean(y4['residual']**2))))
    print('Yamaguchi volume models:',dict(zip(*[x.tolist() for x in np.unique(y4['volume_model'],return_counts=True)])))

In [ ]:
if mode['polarimetric_mode']=='LSAR_FULL_POL':
    def n99(x): return np.maximum(x,0)/(np.nanpercentile(np.maximum(x,0),99)+1e-12)
    fig,ax=plt.subplots(1,3,figsize=(15,4))
    ax[0].imshow(np.dstack([n99(pauli['pauli_3']),n99(pauli['pauli_2']),n99(pauli['pauli_1'])])); ax[0].set_title('Pauli RGB'); ax[0].axis('off')
    ax[1].imshow(np.dstack([n99(fd['double_bounce']),n99(fd['volume']),n99(fd['surface'])])); ax[1].set_title('Freeman RGB: DB/V/S'); ax[1].axis('off')
    ax[2].imshow(np.dstack([n99(y4['double_bounce']),n99(y4['volume']),n99(y4['surface'])])); ax[2].set_title('Yamaguchi RGB: DB/V/S'); ax[2].axis('off')
    plt.tight_layout(); plt.show()